## Importing All  Libraries :

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso,ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from tqdm import tqdm
import joblib
os.environ["JOBLIB_MULTIPROCESSING"] = "0"
os.environ["LOKY_MAX_CPU_COUNT"] = "1"

In [14]:
# Loading the preprocessed data :
file_path=os.path.join(os.path.dirname(os.getcwd()),'data','preprocessed','preprocessed_df.csv')
try:
    if os.path.exists(file_path) is not True :
        raise FileNotFoundError(f'The required file path is not found : {file_path}')
    df=pd.read_csv(file_path)
    print('Dataset Loaded Succesfully ...')
    print(f'Dataset have : {df.shape[0] } rows and {df.shape[1]} columns.')
except FileNotFoundError as e:
    print(e)
except Exception as e:
    print(f'Unexpected error is occcured :{str(e)}')

Dataset Loaded Succesfully ...
Dataset have : 10462 rows and 20 columns.


In [15]:
# Perform target mean encoding for the Airline_cleaned column :
df['log_price'] = np.log1p(df['Price'])

# Drop target and unused columns
X = df.drop(columns=['Price', 'log_price'])
y_log = df['log_price']                   
y_price = df['Price']                      

# Split same for both
X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)
_, _, y_train_price, y_test_price = train_test_split(X, y_price, test_size=0.2, random_state=42)

train_data=X_train.copy()
train_data['Price']=y_train_price

#Compute mean price for each Airline
airline_target_mean =train_data.groupby('Airline_cleaned')['Price'].mean().to_dict()
X_train['Airline_Target_Enco'] =X_train['Airline_cleaned'].map(airline_target_mean)
X_test['Airline_Target_Enco'] =X_test['Airline_cleaned'].map(airline_target_mean)

global_mean = y_train_price.mean()
X_test['Airline_Target_Enco'] =X_test['Airline_Target_Enco'].fillna(global_mean)

# Drop the original categorical column
X_train = X_train.drop(columns='Airline_cleaned')
X_test = X_test.drop(columns='Airline_cleaned')

In [16]:
# Scaling the featuers with mean 0 and std 1 with Standardscalar :
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)


## implementing model building with various regression algorithms :

In [17]:
# Model initialization as dict :
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "ElasticNet": ElasticNet(),
    "SVR": SVR(),
    "DecisionTree": DecisionTreeRegressor(),
    "RandomForest": RandomForestRegressor(),
    "GradientBoosting": GradientBoostingRegressor(),
    "AdaBoost": AdaBoostRegressor(),
    "XGBoost": XGBRegressor(),
    "LightGBM": LGBMRegressor()
}

results = []

for name, model in models.items():
    # Use log_price for linear models and actual Price for tree-based
    if name in ["LinearRegression","Ridge","Lasso","ElasticNet","SVR"]:
        y_train_used = y_train_log
        y_test_used = y_test_log
        model.fit(X_train_scaled, y_train_used)
        train_preds = np.expm1(model.predict(X_train_scaled))  
        test_preds = np.expm1(model.predict(X_test_scaled))    
        y_train_actual = np.expm1(y_train_used)
        y_test_actual = np.expm1(y_test_used)
    else:
        y_train_used = y_train_price
        y_test_used = y_test_price
        model.fit(X_train_scaled, y_train_used)
        train_preds = model.predict(X_train_scaled)
        test_preds = model.predict(X_test_scaled)
        y_train_actual = y_train_price
        y_test_actual = y_test_price

    train_mae = mean_absolute_error(y_train_actual, train_preds)
    train_rmse = np.sqrt(mean_squared_error(y_train_actual, train_preds))
    train_r2 = r2_score(y_train_actual, train_preds)

    test_mae = mean_absolute_error(y_test_actual, test_preds)
    test_rmse = np.sqrt(mean_squared_error(y_test_actual, test_preds))
    test_r2 = r2_score(y_test_actual, test_preds)

    overfit_gap = train_r2 - test_r2


    results.append({
        "Model": name,
        "Train MAE": train_mae,
        "Train RMSE": train_rmse,
        "Train R2": train_r2,
        "Test MAE": test_mae,
        "Test RMSE": test_rmse,
        "Test R2": test_r2,
        "Overfit Gap (R2)": overfit_gap
    })

results_df = pd.DataFrame(results)
results_df.sort_values(by="Test R2", ascending=False, inplace=True)
results_df.reset_index(drop=True, inplace=True)
results_df


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000733 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 129
[LightGBM] [Info] Number of data points in the train set: 8369, number of used features: 19
[LightGBM] [Info] Start training from score 9051.693512


,Model,Train MAE,Train RMSE,Train R2,Test MAE,Test RMSE,Test R2,Overfit Gap (R2)
0,XGBoost,631.502747,1060.670366,0.947718,867.013123,1639.558935,0.871073,0.076645
1,RandomForest,379.304346,847.756480,0.966601,844.190440,1703.321547,0.860851,0.105750
2,LightGBM,895.451425,1574.210690,0.884836,1007.552964,1710.101451,0.859741,0.025095
3,GradientBoosting,1228.949001,1878.072042,0.836086,1252.296568,2014.824770,0.805301,0.030785
4,SVR,1148.596836,1913.741080,0.829801,1213.428580,2041.717604,0.800069,0.029732
5,DecisionTree,187.278454,664.261861,0.979495,979.646389,2083.342667,0.791834,0.187660
6,LinearRegression,2332.638717,16310.036823,-11.362338,1923.048638,3008.452637,0.565915,-11.928253
7,Ridge,2332.322831,16297.201871,-11.342889,1923.070501,3008.523339,0.565895,-11.908784
8,AdaBoost,2814.393401,3400.851715,0.462515,2894.443855,3581.025280,0.384961,0.077555
9,Lasso,3622.374609,4761.624607,-0.053661,3594.317002,4664.005721,-0.043293,-0.010368


###  Model Performance Summary (Before Hyperparameter Tuning) :

The table below summarizes the training and test performance of various regression models evaluated on the dataset:

| Model             | Train R² | Test R² | Overfit Gap (R²) | Train RMSE | Test RMSE | Train MAE | Test MAE |
|------------------|----------|---------|------------------|------------|-----------|-----------|----------|
| XGBoost          | 0.948    | 0.871   | 0.077            | 1060.67    | 1639.56   | 631.50    | 867.01   |
| RandomForest     | 0.967    | 0.861   | 0.106            | 847.76     | 1703.32   | 379.30    | 844.19   |
| LightGBM         | 0.885    | 0.860   | 0.025            | 1574.21    | 1710.10   | 895.45    | 1007.55  |
| GradientBoosting | 0.836    | 0.805   | 0.031            | 1878.07    | 2014.82   | 1228.95   | 1252.30  |
| SVR              | 0.830    | 0.800   | 0.030            | 1913.74    | 2041.72   | 1148.60   | 1213.43  |
| DecisionTree     | 0.979    | 0.792   | 0.188            | 664.26     | 2083.34   | 187.28    | 979.65   |
| LinearRegression | -11.36   | 0.566   | -11.93           | 16310.04   | 3008.45   | 2332.64   | 1923.05  |
| Ridge            | -11.34   | 0.566   | -11.91           | 16297.20   | 3008.52   | 2332.32   | 1923.07  |
| AdaBoost         | 0.463    | 0.385   | 0.078            | 3400.85    | 3581.03   | 2814.39   | 2894.44  |
| Lasso            | -0.054   | -0.043  | -0.010           | 4761.62    | 4664.01   | 3622.37   | 3594.32  |
| ElasticNet       | -0.054   | -0.043  | -0.010           | 4761.62    | 4664.01   | 3622.37   | 3594.32  |

**Key Observations:**
-  **XGBoost**, **RandomForest**, and **LightGBM** perform best on the test set with R² > 0.85.
-  **DecisionTree** shows high overfitting (`Overfit Gap = 0.188`).
-  **LinearRegression**, **Ridge**, **Lasso**, and **ElasticNet** perform poorly with negative or low R² scores.
-  These results are before hyperparameter tuning. Further tuning is expected to improve test performance.



In [18]:
# Tunning the models for better generilization :
param_grid = {
    "XGBoost": {
        "n_estimators": [100, 200],
        "max_depth": [3, 6],
        "learning_rate": [0.05, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "reg_lambda": [1, 3]
    },
    "LightGBM": {
        "n_estimators": [100, 200],
        "max_depth": [3, 6],
        "learning_rate": [0.05, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "reg_alpha": [0, 1]
    },
    "RandomForest": {
        "n_estimators": [100, 200],
        "max_depth": [10, 20],
        "max_features": ['sqrt', 'log2'],
        "min_samples_split": [2, 5]
    },
    "GradientBoosting": {
        "n_estimators": [100, 200],
        "max_depth": [3, 6],
        "learning_rate": [0.05, 0.1],
        "subsample": [0.8, 1.0],
        "min_samples_split": [2, 5]
    },
    "SVR": {
        "kernel": ['rbf'],
        "C": [1, 10],
        "epsilon": [0.1, 0.2],
        "gamma": ['scale', 'auto'],
        "shrinking": [True, False]  
    }
}


models_to_tune = {
    "XGBoost": XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
    "LightGBM": LGBMRegressor(random_state=42, n_jobs=-1),
    "RandomForest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "SVR": SVR()
}


In [19]:
from sklearn.model_selection import GridSearchCV

best_params = {}

for model_name in tqdm(models_to_tune, desc="Tuning Models"):
    model = models_to_tune[model_name]
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid[model_name],
        cv=5,
        scoring='r2',
        n_jobs=-1,
        verbose=0
    )
    
    grid.fit(X_train_scaled,y_train_price if name != "SVR" else y_train_log)
    
    best_params[model_name] = grid.best_params_


Tuning Models:  20%|██        | 1/5 [00:59<03:58, 59.75s/it]

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000718 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 125
[LightGBM] [Info] Number of data points in the train set: 6695, number of used features: 18
[LightGBM] [Info] Start training from score 9042.313816
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

Tuning Models:  40%|████      | 2/5 [02:06<03:11, 63.68s/it]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


Tuning Models: 100%|██████████| 5/5 [10:41<00:00, 128.36s/it]


In [38]:
Tunned_results=[]

for model in tqdm(best_params,desc='Model Processing -'):
    best_model = models_to_tune[model].__class__(**best_params[model])
    if model != 'SVR':
        best_model.fit(X_train_scaled,y_train_price)
        y_pred=best_model.predict(X_test_scaled)
        y_test=y_test_price
    else:
        best_model.fit(X_train_scaled,y_train_log)
        y_pred=np.expm1(best_model.predict(X_test_scaled))
        y_test=np.expm1(y_test_log)
    

    mae = mean_absolute_error(y_test,y_pred)
    rmse = np.sqrt(mean_squared_error(y_test,y_pred))
    r2_Score = r2_score(y_test,y_pred)

    Tunned_results.append({
        "Tunned_Model":'Tunned_'+model,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2_Score
    })

Tunned_results_df = pd.DataFrame(Tunned_results)
Tunned_results_df.sort_values(by="R2", ascending=False, inplace=True)
Tunned_results_df.reset_index(drop=True, inplace=True)
Tunned_results_df

Model Processing -:  20%|██        | 1/5 [00:00<00:01,  2.29it/s]

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000964 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 129
[LightGBM] [Info] Number of data points in the train set: 8369, number of used features: 19
[LightGBM] [Info] Start training from score 9051.693512
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

Model Processing -: 100%|██████████| 5/5 [00:13<00:00,  2.64s/it]


,Tunned_Model,MAE,RMSE,R2
0,Tunned_XGBoost,893.798523,1633.030312,0.872098
1,Tunned_GradientBoosting,899.593802,1657.838385,0.868183
2,Tunned_LightGBM,974.183241,1701.012905,0.861228
3,Tunned_RandomForest,904.970855,1723.292466,0.857568
4,Tunned_SVR,1274.103271,1998.376990,0.808467


###  Tuned Model Performance Summary :

Below is the evaluation summary of the top 5 models **after hyperparameter tuning**, ranked by R² score on the test set:

| Tuned Model            | Test MAE   | Test RMSE  | Test R² |
|------------------------|------------|------------|---------|
| Tunned_XGBoost         | 893.80     | 1633.03    | 0.8721  |
| Tunned_GradientBoosting| 899.59     | 1657.84    | 0.8682  |
| Tunned_LightGBM        | 974.18     | 1701.01    | 0.8612  |
| Tunned_RandomForest    | 904.97     | 1723.29    | 0.8576  |
| Tunned_SVR             | 1274.10    | 1998.38    | 0.8085  |

**Insights:**
-  **Tunned_XGBoost** achieves the highest R² score (0.872) with relatively low MAE and RMSE, making it the best performer.
-  Hyperparameter tuning significantly improved model stability and reduced overfitting compared to untuned versions.
-  **Tunned_SVR**, while improved, still lags behind tree-based models in R² score and error metrics.



In [33]:
filterd_df=results_df[results_df['Test R2']>0.80].drop(columns=['Train MAE','Train RMSE','Train R2','Overfit Gap (R2)'])

In [35]:
filterd_df

,Model,Test MAE,Test RMSE,Test R2
0,XGBoost,867.013123,1639.558935,0.871073
1,RandomForest,844.190440,1703.321547,0.860851
2,LightGBM,1007.552964,1710.101451,0.859741
3,GradientBoosting,1252.296568,2014.824770,0.805301
4,SVR,1213.428580,2041.717604,0.800069


In [39]:
Tunned_results_df

,Tunned_Model,MAE,RMSE,R2
0,Tunned_XGBoost,893.798523,1633.030312,0.872098
1,Tunned_GradientBoosting,899.593802,1657.838385,0.868183
2,Tunned_LightGBM,974.183241,1701.012905,0.861228
3,Tunned_RandomForest,904.970855,1723.292466,0.857568
4,Tunned_SVR,1274.103271,1998.376990,0.808467


In [45]:
Tunned_results_df.to_string

<bound method DataFrame.to_string of               Tunned_Model          MAE         RMSE        R2
0           Tunned_XGBoost   893.798523  1633.030312  0.872098
1  Tunned_GradientBoosting   899.593802  1657.838385  0.868183
2          Tunned_LightGBM   974.183241  1701.012905  0.861228
3      Tunned_RandomForest   904.970855  1723.292466  0.857568
4               Tunned_SVR  1274.103271  1998.376990  0.808467>

In [ ]:
#Combining the two results Dataframes together as final result Dataframe :
filterd_df.columns = ['Model','Test MAE','Test RMSE','Test R2']
Tunned_results_df.columns = ['Model','Tuned MAE','Tuned RMSE','Tuned R2']

# Clean model names in the tuned DataFrame to match
Tunned_results_df['Model'] = Tunned_results_df['Model'].str.replace("Tunned_", "").str.strip()

# Merge both on Model
Final_result_df = pd.merge(Tunned_results_df,filterd_df, on='Model')

# Calculate improvements 
Final_result_df["MAE Change"] = Final_result_df["Test MAE"] - Final_result_df["Tuned MAE"]
Final_result_df["R2 Gain"] = Final_result_df["Tuned R2"] - Final_result_df["Test R2"]

Final_result_df.to_csv(os.path.join(os.path.dirname(os.getcwd()),'data','preprocessed','Final_result_df.csv'),index=False)


In [2]:
import os
os.path.exists(os.path.join(os.path.dirname(os.getcwd()),'data','preprocessed'))

True

In [58]:
#Saving the best model:
best_model=XGBRegressor().__class__(random_state=42, n_jobs=-1, verbosity=0,**best_params['XGBoost'])
try:
    with open('../models/XGB_model.pkl','wb') as file:
        joblib.dump(best_model,file)
except Exception as e:
    raise Exception(f'Ueexpected error occured: {e}')

###  Project Challenges Summary :

During the course of this project, several key challenges were encountered:

1. **Data Quality & Cleaning**  
   - Handling missing values, inconsistent formats, and potential outliers in salary and compensation fields required careful preprocessing.
   - Feature scaling and encoding were necessary due to mixed data types (categorical and numerical).

2. **Feature Engineering Decisions**  
   - Choosing which features to log-transform, drop, or bin was non-trivial and required exploratory analysis and experimentation.
   - Avoiding overengineering while retaining interpretability and model performance balance was important.

3. **Model Selection & Overfitting**  
   - Initial models showed signs of overfitting (especially tree-based models with deep structures).
   - Some linear models failed to generalize well, resulting in negative R² scores.

4. **Hyperparameter Tuning Complexity**  
   - Selecting the right range of parameters for each model without overburdening GridSearchCV was a balancing act.
   - Tuning models with `cv=5` on ~10,000 rows was computationally intensive and time-consuming.

5. **Evaluation & Interpretation**  
   - Choosing the correct evaluation metric (R² score) and interpreting results across different models was crucial for fair comparison.
   - Ensuring reproducibility while running extensive grid searches with tqdm progress feedback helped manage runtime expectations.

 Despite these challenges, structured experimentation, visualization, and iterative tuning led to a robust and well-performing set of models.
